# Transform Circuits Data

1. Read bronze `circuits` table.
2. Keep only the columns required for analytics (Drop `url` column)
3. Standardise column names using snake_case (`circuitId` → `circuit_id`, `circuitName` → `circuit_name`)
4. Rename columns to make them more meaningful (`lat` → `latitude`, `long` → `longitude`)
5. Filter out rows where `circuit_id` is null (business key validation)
6. Remove duplicate records
7. Transform values of columns `circuit_name` and `locality` to Title Case
8. Write the transformed data to silver `circuits` table


#### Step 0 - Load configuration and variables

In [0]:
dbutils.widgets.text("p_batch_id", "")

v_batch_id = dbutils.widgets.get("p_batch_id")
print(v_batch_id)

In [0]:
%run ../00_common/01_configuration

In [0]:
%run ../00_common/03_silver_functions

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"

In [0]:
from pyspark.sql import functions as F

#### Step 1- Read bronze circuits table

In [0]:
circuits_df = (
    spark.table(bronze_table).filter((F.col("batch_id")== v_batch_id ))
    )


In [0]:
# display(circuits_df)

#### Step 2 - Keep only the columns required for analytics (Drop url column)

In [0]:
circuits_selected_df = circuits_df.select(
    F.col("circuitId"),
    F.col("circuitName"),
    F.col("lat"),
    F.col("long"),
    F.col("locality"),
    F.col("country"),
    F.col("ingestion_timestamp"),
    F.col("source_file"),
    F.col("batch_id"),
)

#### Step 3 & 4 - Standardise Column Names
  - Standardise column names using snake_case (`circuitId` → `circuit_id`, `circuitName` → `circuit_name`)
  - Rename columns to make them more meaningful (`lat` → `latitude`, `long` → `longitude`)


In [0]:
circuits_renamed_df = (
    circuits_selected_df
        .withColumnsRenamed({
            "circuitId": "circuit_id",
            "circuitName": "circuit_name",
            "lat": "latitude",
            "long": "longitude"
        })
)

#### Step 5 - Filter out rows where circuit_id is null (business key validation)

In [0]:
# circuits_valid_df = circuits_renamed_df.filter(
#     "circuit_id IS NOT NULL"
# )

In [0]:
circuits_valid_df = circuits_renamed_df.filter(
    F.col("circuit_id").isNotNull()
)

In [0]:
# display(circuits_valid_df)

#### Step 6 - Remove duplicate records

In [0]:
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])

In [0]:
# display(circuits_distinct_df)

#### Step 7 - Transform values of columns `circuit_name` and `locality` to Title Case


In [0]:
circuits_final_df = (
    circuits_distinct_df
        .withColumn('circuit_name', F.initcap(F.col("circuit_name")))
        .withColumn('locality', F.initcap(F.col("locality")))
)

In [0]:
# display(circuits_final_df)

#### Step 8 - Write the transformed data to silver `circuits` table

In [0]:
write_to_silver(
    input_df=circuits_final_df,
    target_table=silver_table,
    merge_condition="t.circuit_id = s.circuit_id",
    columns_to_update=[
        "circuit_name",
        "latitude",
        "longitude",
        "locality",
        "country",
        "ingestion_timestamp",
        "source_file",
        "batch_id"        
    ]
)

In [0]:
# display(spark.table(silver_table))